In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:

def evaluate_uncertainty(prob, strategy):
    """Evaluate the desired uncertainty sampling strategy on predictive
    probabilities 'prob'.

    PARAMETERS
    ----------
    prob : ndarray 
        numpy array with predictive probabilities of shape 
        (n_points, n_classes)
    strategy : str
        One of 'least confident', 'margin', or 'entropy'.

    The function should return an array with uncertainties of shape
    (n_points, ) corresponding to the desired strategy.
    """
    if strategy == 'least confident':
        return 1 - np.max(prob,axis = 1)
    elif strategy == 'margin':
        prob = np.sort(prob, axis = 1)
        return 1 - (prob[:,-1] - prob[:,-2])
    elif strategy == 'entropy':
        entropy = - prob * np.log2(prob)
        return np.sum(entropy, axis = 1)

In [ ]:

model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)
model.fit(X_train, y_train)

# 4. Make predictions
y_pred = model.predict(X_test)

# 5. Evaluate
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy:", accuracy)

In [ ]:
prob = model.predict_proba()
least_confident = evaluate_uncertainty(prob, 'least confident')
margin = evaluate_uncertainty(prob, 'margin')
entropy = evaluate_uncertainty(prob, 'entropy')


In [ ]:
order = np.random.permutation(range(len(Xpool)))


#samples in the pool
poolidx=np.arange(len(Xpool),dtype='int')
ninit = 10 #initial samples
#initial training set
trainset=order[:ninit]
poolidx=np.arange(len(Xpool),dtype=int)
poolidx=np.setdiff1d(poolidx,trainset)

Xtrain=np.take(Xpool,trainset,axis=0)
ytrain=np.take(ypool,trainset,axis=0)
#remove data from pool
poolidx=np.setdiff1d(poolidx,trainset)
testacc_al = []
addn = 5

for i in range(25):
    data = np.take(Xpool,trainset,axis=0)
    labels = np.take(ypool,trainset,axis=0)
    model.fit(data, labels)
    #predict and calculate the accuracy
    ypred = model.predict(Xtest)

    #calculate accuracy on test set
    accuracy = sum(ytest == ypred)/len(ytest)
    testacc_al.append((ninit+i*addn,accuracy)) #add in the accuracy
    print('Model: LR, %i random samples'%(ninit+i*addn))

    #find next index (with highest uncertainty)
    prop = model.predict_proba(np.take(Xpool,poolidx,axis=0)) # find uncertainty

    
    top_indices = poolidx[np.argsort(1-np.max(prop, axis =1))[-addn:]]
    print(top_indices)
    trainset = np.append(trainset, top_indices)
    poolidx = np.setdiff1d(poolidx,trainset)